In [1]:
from pageindex import PageIndexClient
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import os

In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [3]:
pageindex_client = PageIndexClient(api_key=os.getenv("PAGEINDEX_API_KEY"))
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
PDF_URL = "https://arxiv.org/pdf/1706.03762"
data_dir = "./data"
PDF_PATH = Path(data_dir) / "attention_is_all_you_need.pdf"

In [5]:
print("Uploading the PDF document to PageIndex...")
result = pageindex_client.submit_document(PDF_PATH)

doc_id = result["doc_id"]
print(f"Document uploaded with doc_id: {doc_id}")

Uploading the PDF document to PageIndex...
Document uploaded with doc_id: pi-cmnq7vdqx02op01pgdvnf1w07


In [7]:
print("Building tree index... ")
import time

while True:
    status_result = pageindex_client.get_document(doc_id)
    status = status_result["status"]
    print(f"Current document status: {status}")

    if status == "completed":
        print("Document processing completed.")
        break
    elif status == "failed":
        print("Document processing failed.")
        exit(1)
    else:
        print("Document is still processing. Waiting for 10 seconds before checking again...")
        time.sleep(10)

Building tree index... 
Current document status: processing
Document is still processing. Waiting for 10 seconds before checking again...
Current document status: completed
Document processing completed.


In [8]:
import json
tree_result = pageindex_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result",[])

print(f"Top-level sections: {len(pageindex_tree)}")
print("Raw tree structure (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

# Optional: save locally
import json
with open(Path(data_dir) / "attention_is_all_you_need.json", "w") as f:
    json.dump(pageindex_tree, f)

Top-level sections: 1
Raw tree structure (first node):
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Attention Is All You Need\n\nAshish Vaswani*\n\nGoogle Brain\n\navaswani@google.com\n\nNoam Shazeer*\n\nGoogle Brain\n\nnoam@google.com\n\nNiki Parmar*\n\nGoogle Research\n\nnikip@google.com\n\nJakob Uszkoreit*\n\nGoogle Research\n\nusz@google.com\n\nLlion Jones*\n\nGoogle Research\n\nllion@google.com\n\nAidan N. Gomez*\u2020\n\nUniversity of Toronto\n\naidan@cs.toronto.edu\n\n\u0141ukasz Kaiser*\n\nGoogle Brain\n\nlukaszkaiser@google.com\n\nIllia Polosukhin*\u2021\n\nillia.polosukhin@gmail.com\n",
  "text": "# Attention Is All You Need\n\nAshish Vaswani*\n\nGoogle Brain\n\navaswani@google.com\n\nNoam Shazeer*\n\nGoogle Brain\n\nnoam@google.com\n\nNiki Parmar*\n\nGoogle Research\n\nnikip@google.com\n\nJakob Uszkoreit*\n\nGoogle Research\n\nusz@google.com\n\nLlion Jones*\n\nGoogle Research\n\nllion@google.com\n\nAidan N. Gomez*\u

In [9]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent + ("--" if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

In [10]:
print_tree(pageindex_tree)

[0000] Attention Is All You Need (p.1)
  --[0001] Abstract (p.1)
  --[0002] 2 Background (p.2)
  --[0003] 3 Model Architecture (p.2)
    --[0004] 3.1 Encoder and Decoder Stacks (p.3)
    --[0005] 3.2 Attention (p.3)
      --[0006] 3.2.1 Scaled Dot-Product Attention (p.4)
      --[0007] 3.2.2 Multi-Head Attention (p.4)
      --[0008] 3.2.3 Applications of Attention in our Model (p.5)
    --[0009] 3.3 Position-wise Feed-Forward Networks (p.5)
    --[0010] 3.4 Embeddings and Softmax (p.5)
    --[0011] 3.5 Positional Encoding (p.6)
  --[0012] 4 Why Self-Attention (p.6)
  --[0013] 5 Training (p.7)
  --[0014] 6 Results (p.8)
    --[0015] 6.1 Machine Translation (p.8)
    --[0016] 6.2 Model Variations (p.8)
    --[0017] 6.3 English Constituency Parsing (p.9)
  --[0018] 7 Conclusion (p.10)
  --[0019] References (p.10)
  --[0020] Attention Visualizations (p.13)


In [11]:
def count_nodes(nodes):
    count = 0
    for node in nodes:
        count += 1
        if node.get("nodes"):
            count += count_nodes(node["nodes"])
    return count

In [12]:
total_nodes = count_nodes(pageindex_tree)
print(f"Total nodes in the tree: {total_nodes}")

Total nodes in the tree: 21


In [ ]:
def get_relevant_nodes(query: str, tree: list, model: str = "gpt-4o") -> dict:
    """
    Identifies relevant document sections for a given query using LLM reasoning.

    Args:
        query: User's question or search query
        tree: Hierarchical document structure from PageIndex
        model: OpenAI model to use for section identification

    Returns:
        Dictionary containing LLM reasoning and list of relevant node IDs
    """

    # Build simplified tree representation for LLM analysis
    def build_simplified_structure(node_list, depth=0):
        simplified = []
        for item in node_list:
            node_info = {
                "node_id": item["node_id"],
                "title": item["title"],
                "page_index": item.get("page_index", "?")
            }

            # Add text preview if available for better context
            content = item.get("text", "")
            if content:
                node_info["summary"] = content[:200]

            # Recursively process child nodes
            if "nodes" in item and item["nodes"]:
                node_info["children"] = build_simplified_structure(item["nodes"], depth + 1)

            simplified.append(node_info)

        return simplified

    # Create lightweight tree structure
    simplified_tree = build_simplified_structure(tree)

    # Construct analysis prompt
    analysis_prompt = f"""Analyze the document structure below and identify sections relevant to answering the user's query.

User Query: {query}

Document Structure (hierarchical table of contents):
{json.dumps(simplified_tree, indent=2)}

Instructions:
1. Reason through which sections would help answer the query
2. Return your analysis in JSON format with:
   - "thinking": your reasoning process
   - "node_list": array of relevant node_id values

Required JSON response format:
{{
  "thinking": "your step-by-step analysis",
  "node_list": ["id1", "id2"]
}}
"""

    # Get LLM analysis of relevant sections
    completion = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": analysis_prompt}],
        response_format={"type": "json_object"}
    )

    # Parse and return the structured response
    analysis_result = json.loads(completion.choices[0].message.content)
    return analysis_result

In [ ]:
query = "How does the Transformer encode positional information, and what are the mathematical functions used?"

print(f"Running LLM Tree Search for query: '{query}'")
result = get_relevant_nodes(query, pageindex_tree)
print("LLM Tree Search Result:")
print(result.get("thinking", "N/A"))
print()
print(f"Relevant node IDs: {result.get('node_list', [])}")

In [15]:
def find_nodes_by_ids(tree: list, target_ids:list) -> list:
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [16]:
def generate_answer(query: str, nodes: list, model: str = "gpt-4o") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer to the query.
    Instructs the LLM to cite section titles and page numbers.
    """

    if not nodes:
        return "No relevant sections found to answer the query."

    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}] \n"
            f"{node.get('text', 'Content not available')}"
        )
    context = "\n\n--\n\n".join(context_parts)

    prompt = f"""Your are an expert document analyst. Answer the question using Only the provided context.
For every claim you make, cite the section title and page number in parantheses. If the answer is not in the context, say you don't know.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:
"""
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role":"user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

In [ ]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG Pipeline:
    
    Step 1: LLM Tree search to find relevant node_ids
    Step 2: Find nodes in the tree matching those node_ids
    Step 3: Generate an answer grounded in the retrieved nodes, with citations
    """

    if verbose:
        print("*" * 50)
        print(f"Query: {query}")
        print("*" * 50)
        

    search_result = get_relevant_nodes(query, tree)
    node_ids = search_result.get("node_list", [])

    if verbose:
        print("LLM Tree Search Reasoning:")
        print(f"{search_result.get('thinking', '')[:200]}...")
        print(f"Relevant node IDs: {node_ids}")
        print("-" * 50)

    relevant_nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"Retrieved {len(relevant_nodes)} relevant nodes for answer generation.")
        for node in relevant_nodes:
            print(f"- {node['title']} (p.{node.get('page_index', '?')})")
    answer = generate_answer(query, relevant_nodes)

    if verbose:
        print("-" * 50)
        print("Generated Answer:")
        print(answer)
    return answer

In [18]:
answer = vectorless_rag(query, pageindex_tree)
print("\nFinal Answer:")
print(answer)

**************************************************
Query: How does the Transformer encode positional information, and what are the mathematical functions used?
**************************************************
LLM Tree Search Reasoning:
To address the query about how the Transformer encodes positional information and the mathematical functions used, we need to locate sections related to positional encoding. The query specifically ask...
Relevant node IDs: ['0011']
--------------------------------------------------
Retrieved 1 relevant nodes for answer generation.
- 3.5 Positional Encoding (p.6)
--------------------------------------------------
Generated Answer:
The Transformer encodes positional information using sine and cosine functions of different frequencies. The mathematical functions used are:

- \( PE_{(pos, 2i)} = \sin(pos / 10000^{2i / d_{\text{model}}}) \)
- \( PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i / d_{\text{model}}}) \)

where \( pos \) is the position and \( i \) is t

In [19]:
query = "What does the term “self-attention” mean?"

In [20]:
answer = vectorless_rag(query, pageindex_tree)
print("\nFinal Answer:")
print(answer)

**************************************************
Query: What does the term “self-attention” mean?
**************************************************
LLM Tree Search Reasoning:
To determine the meaning of the term 'self-attention', I need to locate sections in the document that discuss this specific term or provide a description or definition. Starting with the document tree...
Relevant node IDs: ['0004', '0005', '0012']
--------------------------------------------------
Retrieved 3 relevant nodes for answer generation.
- 3.1 Encoder and Decoder Stacks (p.3)
- 3.2 Attention (p.3)
- 4 Why Self-Attention (p.6)
--------------------------------------------------
Generated Answer:
The term "self-attention" refers to a mechanism in which each position in a sequence computes a weighted sum of the values, with weights computed using a compatibility function over the queries and keys, allowing the model to connect all positions with a constant number of sequential operations. This can facilita

In [ ]:
query = "What are the two main parts of the Transformer model?"

In [ ]:
answer = vectorless_rag(query, pageindex_tree)
print("\nFinal Answer:")
print(answer)

AttributeError: 'dict' object has no attribute 'choices'

In [34]:
answer = response['choices'][0]['message']['content'].strip()
print("Answer from PageIndex API:")
print(answer)

Answer from PageIndex API:
I'll get the document structure first to locate the key sections explaining Attention Residuals.## Attention Residuals in Simple Language

**The Problem with Standard Residuals:**

In deep neural networks, residual connections add together outputs from all previous layers with equal weight (like 1+1+1+1...). This creates issues:
- Each layer's contribution gets diluted as the network gets deeper
- Later layers can't selectively retrieve useful information from earlier layers
- Hidden states grow uncontrollably, destabilizing training

**The Attention Residuals Solution:**

Instead of blindly adding all layer outputs equally, **Attention Residuals lets each layer *choose* which previous layers to pay attention to**. It's like the difference between:
- **Standard**: "Mix everything together equally"
- **AttnRes**: "Look at all previous layers and decide which ones are most relevant right now"

**The Key Analogy:**

Think of how transformers handle sequences of 

In [46]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

current = None

stream = await async_client.responses.create(
    model="gpt-5.4",
    stream = True,
    reasoning = {"effort": "medium", "summary": "auto"},
    instructions = "You are a document analysis assistant. Always use the available tools to search and retrieve information from the user's documents. Start by finding relevant documents, then get their structure and content.",
    tools = [
        {
            "type": "mcp",
            "server_label": "PageIndex",
            "server_url": "https://api.pageindex.ai/mcp",
            "headers": {"Authorization": f"Bearer {os.getenv('PAGEINDEX_API_KEY')}"},
            "require_approval": "never",
        }
    ],
    input = query
)

async for event in stream:
    if event.type == "response.output_item.added":
        if event.item.type == "reasoning":
            current = "reasoning"
            print("[reasoning] ", end="", flush=True)
        elif event.item.type == "mcp_list_tools":
            print(f"Discovered MCP tools\n")
    elif event.type == "response.reasoning_summary_part.done":
        print("...\n")
    elif event.type == "response.reasoning_summary_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.output_item.done":
        if event.item.type == "mcp_call":
            current = "tool"
            print(f"[tool_use] {event.item.name} ({event.item.arguments})\n")
            print(f"[tool result] {str(event.item.output)[:200]}...\n")
    elif event.type == "response.output_text.delta":
        if current != "answer":
            current = "answer"
            print("[answer]")
        print(event.delta, end="", flush=True)

Discovered MCP tools

[reasoning] [answer]
“Attention residuals” usually refers to the **skip connection** around an attention layer in a transformer.

Simple way to think about it:

- The model has some current understanding of the text.
- The **attention layer** adds new information by looking at other words.
- The **residual connection** says:  
  **don’t throw away the old understanding — keep it, and add the new attention output on top.**

In equation form, it’s basically:

**new state = old state + attention output**

### Why do this?
Because it helps the model:

- **keep useful original information**
- **learn small improvements instead of rebuilding everything**
- **train more easily and more stably**

### Analogy
Imagine you’re editing a draft:

- **Old draft** = current representation
- **Editor’s suggestions** = attention output
- **Residual connection** = keep the draft and apply the suggestions, instead of rewriting from scratch

So attention residuals are just the model’s